# 03 — Discovery and Metadata

This notebook demonstrates how to discover available stellar atmosphere catalogs and inspect their metadata using the `SED_Tools` Python API.

**Key classes:** `SED`, `CatalogInfo`

In [1]:
from sed_tools.api import SED
import pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib64/python3.13/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib64/python3.13/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/usr/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3.13/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 739, 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## 1. Querying Available Catalogs

`SED.query()` lists all available catalogs — both locally installed ones and those available from remote sources (SVO, MSG, MAST, NJM mirror). Each result is a `CatalogInfo` object.

In [3]:
catalogs = SED.query()

df = pd.DataFrame([{
    'Name':       c.name,
    'Source':     c.source,
    'Teff Range': c.teff_range,
    'logg Range': c.logg_range,
    'Met Range':  c.metallicity_range,
    'N Spectra':  c.n_spectra,
    'Local':      c.is_local,
} for c in catalogs])

df.head(10)

Discovering available models from SVO...
Discovering available models from MSG grids...
Discovering available models from MAST (BOSZ 2024)...


,Name,Source,Teff Range,logg Range,Met Range,N Spectra,Local
0,Husfeld,local,"(35000.0, 80000.0)","(4.0, 7.0)","(999.9, 999.9)",448.0,True
1,Kurucz,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-2.5, 0.5)",3808.0,True
2,Kurucz2003,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-2.5, 0.5)",3808.0,True
3,Kurucz2003all,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-4.0, 0.5)",8092.0,True
4,Kurucz2003all__alpha_0.4__lh_1.25__vtur_2,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-4.0, 0.5)",4284.0,True
5,Kurucz2003all__alpha_00,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-2.5, 0.5)",3808.0,True
6,Kurucz2003all__alpha_04,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-4.0, 0.5)",4284.0,True
7,Kurucz2003all__alpha_0__lh_1.25__vtur_2,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-2.5, 0.5)",3808.0,True
8,Kurucz2003alp,local,"(3500.0, 50000.0)","(0.0, 5.0)","(-4.0, 0.5)",4284.0,True
9,NextGen2,local,"(900.0, 10000.0)","(3.0, 6.0)","(-4.0, 0.0)",2712.0,True


## 2. Filtering the Query

You can narrow by source, or by parameter coverage. Catalogs with at least partial overlap with your requested range are returned.

In [4]:
# Local catalogs only
local_only = SED.query(include_remote=False)
print(f"Locally installed: {[c.name for c in local_only]}")

# Remote SVO only
svo_catalogs = SED.query(source='svo', include_local=False)
print(f"Available from SVO: {len(svo_catalogs)} catalogs")

# Filter by parameter coverage — solar-like stars
solar_like = SED.query(
    teff_min=5000,
    teff_max=7000,
    logg_min=3.5,
    metallicity_min=-1.0,
)
print(f"Covering solar-like parameters: {len(solar_like)} catalogs")

Locally installed: ['Husfeld', 'Kurucz', 'Kurucz2003', 'Kurucz2003all', 'Kurucz2003all__alpha_0.4__lh_1.25__vtur_2', 'Kurucz2003all__alpha_00', 'Kurucz2003all__alpha_04', 'Kurucz2003all__alpha_0__lh_1.25__vtur_2', 'Kurucz2003alp', 'NextGen2', 'bbody', 'hres', 'morley12', 'tmap2']
Discovering available models from SVO...
Available from SVO: 70 catalogs
Discovering available models from SVO...
Discovering available models from MSG grids...
Discovering available models from MAST (BOSZ 2024)...
Covering solar-like parameters: 114 catalogs


## 3. Inspecting CatalogInfo

Each `CatalogInfo` object exposes the catalog's parameter coverage and can answer coverage questions directly.

In [5]:
# Use the first local catalog
info = next(c for c in catalogs if c.is_local)

print(f"Name:              {info.name}")
print(f"Source:            {info.source}")
print(f"Teff range:        {info.teff_range} K")
print(f"logg range:        {info.logg_range}")
print(f"Metallicity range: {info.metallicity_range} [M/H]")
print(f"N spectra:         {info.n_spectra}")
print(f"Locally installed: {info.is_local}")

Name:              Husfeld
Source:            local
Teff range:        (35000.0, 80000.0) K
logg range:        (4.0, 7.0)
Metallicity range: (999.9, 999.9) [M/H]
N spectra:         448
Locally installed: True


### Coverage Checks

`covers()` checks whether a specific point in parameter space is within the catalog. `covers_range()` checks for at least partial overlap with a range.

In [6]:
# Does the catalog include solar parameters?
print(f"Covers the Sun (Teff=5777, logg=4.44)? {info.covers(teff=5777, logg=4.44)}")

# Does it overlap the 5000–6000 K range?
print(f"Overlaps 5000–6000 K?                  {info.covers_range(teff_min=5000, teff_max=6000)}")

# Does it reach below 3000 K?
print(f"Covers T < 3000 K?                     {info.covers_range(teff_max=3000)}")

Covers the Sun (Teff=5777, logg=4.44)? False
Overlaps 5000–6000 K?                  False
Covers T < 3000 K?                     False


## 4. Finding the Best Catalog for a Set of Parameters

`SED.find_model()` ranks locally installed models by proximity to a requested point in parameter space.

In [7]:
sed = SED()
matches = sed.find_model(teff=5777, logg=4.44, metallicity=0.0)

print("Best local models for solar parameters:")
for m in matches[:5]:
    print(f"  {m.name:25s}  Teff {m.teff_range}  logg {m.logg_range}  [M/H] {m.metallicity_range}")

Best local models for solar parameters:
  hres                       Teff (3000.0, 55000.0)  logg (-0.5, 5.5)  [M/H] (-10.0, 3.0)
  Kurucz                     Teff (3500.0, 50000.0)  logg (0.0, 5.0)  [M/H] (-2.5, 0.5)
  Kurucz2003                 Teff (3500.0, 50000.0)  logg (0.0, 5.0)  [M/H] (-2.5, 0.5)
  Kurucz2003all              Teff (3500.0, 50000.0)  logg (0.0, 5.0)  [M/H] (-4.0, 0.5)
  Kurucz2003all__alpha_0.4__lh_1.25__vtur_2  Teff (3500.0, 50000.0)  logg (0.0, 5.0)  [M/H] (-4.0, 0.5)
